In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt
from numpy.polynomial import Polynomial

In [ ]:
max_altitude = 2800.0
max_velocity = 260.0

c_g = 9.8085
c_m = 9.820
c_rho0 = 1.225
c_Cd0 = 0.461
c_A = 0.01247

### PyTorch setup

In [ ]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Availability: {torch.cuda.is_available()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

torch.manual_seed(42)

### Data generation

In [ ]:
n_sims = 10000
n_collocation = 1000000
n_boundary = 10000

sim_dt = 0.1

base_accel_min = 30.0
base_accel_max = 50.0
base_accel_time_min = 2.0
base_accel_time_max = 7.0


cd_df = pd.read_csv(os.path.join(os.path.abspath(''), "data", "taipan_or_data.csv"))
cd_df = cd_df[cd_df["Vertical velocity (m/s)"] > 0.1]
cd_df = cd_df[cd_df["# Time (s)"] > 6.44]

cd_mach = cd_df["Mach number ()"].values
cd_data = cd_df["Drag coefficient ()"].values
cd_p = Polynomial.fit(cd_mach, cd_data, 3)
cd_coeffs = cd_p.convert().coef
cd_weights = torch.tensor(cd_coeffs, dtype=torch.float32, device=device)
cd_degrees = torch.arange(len(cd_weights), dtype=torch.float32, device=device)


def get_cd(h, v):
    a = torch.sqrt(1.4 * 287.05 * (288.15 - 0.0065 * h))
    mach = v / a
    X_poly = mach.unsqueeze(1).pow(cd_degrees)
    y_tensor_pred = X_poly @ cd_weights

    return y_tensor_pred
    

def compute_acc(h, v):
    dynamic_cd = get_cd(h, v)
    dynamic_rho = c_rho0 * torch.pow(1 - 0.0065 * h / 288.15, 4.25588)
    return -c_g - (0.5 * dynamic_rho * v**2 * dynamic_cd * c_A) / c_m


def rocket_derivatives(t, h, v, thrust_accel, burnout_time):
    normal_acc = compute_acc(h, v)
    is_powered = t <= burnout_time
    a_net = torch.where(is_powered, normal_acc + thrust_accel, normal_acc)
    
    return v, a_net


def simulate_flight(initial_h, initial_v, thrust_accel, burnout_time, dt):
    t = 0.0    
    N = initial_h.shape[0]
    h = initial_h
    v = initial_v
    trajectory_cache = [(h.clone(), v.clone())]
    mask_cache = [torch.ones(N, dtype=torch.bool, device=device)]
    active_mask = torch.ones(N, dtype=torch.bool, device=device)
    
    while active_mask.any():
        k1_h, k1_v = rocket_derivatives(t, h, v, thrust_accel, burnout_time)
        
        h2, v2 = h + 0.5 * dt * k1_h, v + 0.5 * dt * k1_v
        k2_h, k2_v = rocket_derivatives(t + 0.5 * dt, h2, v2, thrust_accel, burnout_time)
        
        h3, v3 = h + 0.5 * dt * k2_h, v + 0.5 * dt * k2_v
        k3_h, k3_v = rocket_derivatives(t + 0.5 * dt, h3, v3, thrust_accel, burnout_time)
        
        h4, v4 = h + dt * k3_h, v + dt * k3_v
        k4_h, k4_v = rocket_derivatives(t + dt, h4, v4, thrust_accel, burnout_time)
        
        next_h = h + (dt / 6.0) * (k1_h + 2*k2_h + 2*k3_h + k4_h)
        next_v = v + (dt / 6.0) * (k1_v + 2*k2_v + 2*k3_v + k4_v)
        
        t += dt
        active_mask = (t <= burnout_time) | (next_v > 0)
        
        h = torch.where(active_mask, next_h, h)
        v = torch.where(active_mask, next_v, v)
        
        trajectory_cache.append((h.clone(), v.clone()))
        mask_cache.append(active_mask.clone())

    # Stack into 3D tensors: (TimeSteps, N, Features)
    step_tensors = [torch.stack([hs, vs], dim=-1) for hs, vs in trajectory_cache]
    trajectory_3d = torch.stack(step_tensors, dim=0) 
    masks_2d = torch.stack(mask_cache, dim=0)        
    
    # The apogee is the final recorded height for each rocket
    apogees = trajectory_3d[-1, :, 0] 

    total_steps = trajectory_3d.shape[0]
    time_steps = torch.arange(total_steps, device=device) * dt
    
    # Broadcast comparison: (total_steps, 1) > (1, N) results in (total_steps, N) mask
    coasting_mask = time_steps.unsqueeze(1) > burnout_time.unsqueeze(0)
    
    # Filter and Flatten (This is where the lists of varying lengths are built)
    X_list = []
    Y_list = []
    
    for i in range(N):
        # Boolean mask of valid timesteps for rocket i
        valid_steps = masks_2d[:, i] & coasting_mask[:, i]
        
        # Extract only the active flight path: Shape (L_i, 2)
        X_i = trajectory_3d[valid_steps, i, :]
        
        # Create a matching label tensor filled with this rocket's apogee: Shape (L_i, 1)
        Y_i = torch.full((X_i.shape[0], 1), apogees[i].item(), device=device)
        
        X_list.append(X_i)
        Y_list.append(Y_i)

    return X_list, Y_list


def generate_data():
    thrust_accel = torch.empty(n_sims, device=device).uniform_(base_accel_min, base_accel_max)
    burnout_time = torch.empty(n_sims, device=device).uniform_(base_accel_time_min, base_accel_time_max)
    N = thrust_accel.shape[0]
    h = torch.zeros(N, dtype=torch.float32, device=device)
    v = torch.zeros(N, dtype=torch.float32, device=device)
    
    X_list, Y_list = simulate_flight(h, v, thrust_accel, burnout_time, sim_dt)
    
    # Concatenate lists of varying lengths into the final giant tensors
    X = torch.cat(X_list, dim=0)
    Y = torch.cat(Y_list, dim=0)
    
    # Extract vectors and apply Normalization
    h = X[:, 0].unsqueeze(1) / max_altitude
    v = X[:, 1].unsqueeze(1) / max_velocity
    apogee = Y / max_altitude
    
    return h, v, apogee


# Boundary conditions: at apogee, velocity should be 0 + Normalization
def generate_boundary_points():
    h = torch.rand((n_boundary, 1))
    v = torch.zeros((n_boundary, 1))
    apogee = h

    return h, v, apogee


# Data generation for training the PINN + Normalization
def generate_collocation_points():
    h = torch.rand((n_collocation, 1))
    v = torch.rand((n_collocation, 1))

    return h, v

In [ ]:
# Data preparation
h_data, v_data, y_data = generate_data()
h_boundary, v_boundary, y_boundary = generate_boundary_points()
h_colloc, v_colloc = generate_collocation_points()

h_data = h_data.to(device)
v_data = v_data.to(device)
y_data = y_data.to(device)
h_boundary = h_boundary.to(device)
v_boundary = v_boundary.to(device)
y_boundary = y_boundary.to(device)
h_colloc = h_colloc.to(device)
v_colloc = v_colloc.to(device)

h_colloc.requires_grad_(True)
v_colloc.requires_grad_(True)

print(f"Data length: {len(h_data)}")

### Model

Note: This architecture is made smaller for performance. "Bigger" version consists of layers with 32 neurons instead of 16.

In [ ]:
# Define the Physics-Informed Neural Network (PINN) architecture for airbrake control
class AirbrakePINN(nn.Module):
    def __init__(self):
        super(AirbrakePINN, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(2, 16),
            nn.Tanh(),
            nn.Linear(16, 32),
            nn.Tanh(),
            nn.Linear(32, 16),
            nn.Tanh(),
            nn.Linear(16, 1),
        )

    def forward(self, h, v):
        x = torch.cat([h, v], dim=1)
        return self.net(x)

In [ ]:
# Training
w_data = 2.0
w_boundary = 50.0
w_physics = 1e-4


model = AirbrakePINN().to(device)
losses = {
    "total": [],
    "data": [],
    "boundary": [],
    "physics": []
}


def train_step():
    # Flight data
    outputs_data = model(h_data, v_data)
    loss_data = torch.mean((outputs_data - y_data)**2)

    # Boundary points loss
    outputs_boundary = model(h_boundary, v_boundary)
    loss_boundary = torch.mean((outputs_boundary - y_boundary)**2)

    # Collocation points loss
    outputs_colloc = model(h_colloc, v_colloc)
    dh_dx_norm = torch.autograd.grad(outputs_colloc, h_colloc, grad_outputs=torch.ones_like(outputs_colloc), create_graph=True)[0]
    dh_dv_norm = torch.autograd.grad(outputs_colloc, v_colloc, grad_outputs=torch.ones_like(outputs_colloc), create_graph=True)[0]
    dh_dx_real = dh_dx_norm
    dh_dv_real = dh_dv_norm * (max_altitude / max_velocity)
    h_real = h_colloc * max_altitude
    v_real = v_colloc * max_velocity
    a_real = compute_acc(h_real, v_real)    
    residual = (dh_dx_real * v_real) + (dh_dv_real * a_real)
    loss_physics = torch.mean(residual**2)

    # Total loss
    total_loss = (w_data * loss_data) + (w_boundary * loss_boundary) + (w_physics * loss_physics)
    total_loss.backward()

    losses["total"].append(total_loss.item())
    losses["data"].append(loss_data.item())
    losses["boundary"].append(loss_boundary.item())
    losses["physics"].append(loss_physics.item())

    return total_loss


def print_train_status(epoch, n_epochs, optimizer_name):
    if (epoch + 1) % 100 == 0:
        print(f"[{optimizer_name}] Epoch {epoch+1}/{n_epochs} | Total: {losses['total'][-1]*max_altitude:.7f} | "
              f"Data: {losses['data'][-1]*max_altitude:.7f} | "
              f"Physics: {losses['physics'][-1]:.7f} | "
              f"Boundary: {losses['boundary'][-1]*max_altitude:.7f}"
              )

In [ ]:
model.train()

n_epochs_adam = 15000
adam_optim = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(n_epochs_adam):
    if epoch >= 3000:
        w_physics = 1e-3
        
    adam_optim.zero_grad()
    train_step()
    adam_optim.step()

    print_train_status(epoch, n_epochs_adam, "Adam")

In [ ]:
model.train()

n_epochs_lbfgs = 10000
n_history_size_lbfgs = 100
lbfgs_optim = torch.optim.LBFGS(model.parameters(), lr=0.1, max_iter=n_epochs_lbfgs, history_size=n_history_size_lbfgs)
lbfgs_iter = 0

def closure():
    global lbfgs_iter
    
    lbfgs_optim.zero_grad()
    loss = train_step()

    lbfgs_iter += 1

    print_train_status(lbfgs_iter, n_epochs_lbfgs, "L-BFGS")
    
    return loss

lbfgs_optim.step(closure)

In [ ]:
# Save model
total_params = sum(p.numel() for p in model.parameters())
print(f"Saving model. Total model parameters: {total_params}")

torch.save(model.state_dict(), os.path.join(os.path.abspath(''), "airbrake_pinn.pt"))

### Inference

In [ ]:
# Load model
model = AirbrakePINN().to(device)
model.load_state_dict(torch.load(os.path.join(os.path.abspath(''), "airbrake_pinn.pt"), map_location=device))

In [ ]:
# Get OR data for inference
df = pd.read_csv(os.path.join(os.path.abspath(''), "..", "..", "..", "..", "..", "sim", "hub", "data", "OR_TAIPAN.csv"))
apogee_df = df['Altitude (m)'].max()
df['y'] = apogee_df # Every sample have the same final apogee
df = df.iloc[2600:6165]
t_infer = df['# Time (s)'].values
h_infer = df['Altitude (m)'].values
v_infer = df['Vertical velocity (m/s)'].values
a_infer = df['Vertical acceleration (m/s²)'].values
y_infer = df['y'].values

h_infer_tensor = torch.tensor(h_infer, dtype=torch.float32, device=device)
v_infer_tensor = torch.tensor(v_infer, dtype=torch.float32, device=device)

# Plot height
plt.figure(figsize=(10, 5))
plt.plot(t_infer, h_infer)
plt.xlabel("Time [s]")
plt.ylabel("Altitude [m]")
plt.show()

In [ ]:
# Accelerations
a_aero_infer = torch.tensor(a_infer, dtype=torch.float32, device=device)
a_pred_physics = compute_acc(h_infer_tensor, v_infer_tensor)

plt.plot(t_infer, a_aero_infer.cpu())
plt.plot(t_infer, a_pred_physics.cpu())
plt.xlabel("Time [s]")
plt.ylabel("Acceleration [m/s2]")
plt.show()

In [ ]:
# Simple inference of OR data
model.eval()

with torch.no_grad():
    h_infer_tensor_tmp = torch.tensor(h_infer / max_altitude, dtype=torch.float32).unsqueeze(1).to(device)
    v_infer_tensor_tmp = torch.tensor(v_infer / max_velocity, dtype=torch.float32).unsqueeze(1).to(device)
    y_pred_tensor = model(h_infer_tensor_tmp,v_infer_tensor_tmp)

# Convert to numpy on CPU
y_pred = y_pred_tensor.cpu().numpy()[:, 0] * max_altitude

In [ ]:
# Calculate true RK4 apogees
N = h_infer_tensor.shape
infer_thrust_accel = torch.zeros(N, dtype=torch.float32, device=device)
infer_thrust_time = torch.zeros(N, dtype=torch.float32, device=device)

_, apogees = simulate_flight(h_infer_tensor, v_infer_tensor, infer_thrust_accel, infer_thrust_time, 0.01)

# Format apogees
y_pred_physics = []
for i in range(len(apogees)):
    y_pred_physics.append(apogees[i][0][0].cpu())
y_pred_physics = np.array(y_pred_physics)

In [ ]:
# Draw apogees plot
plt.figure(figsize=(10, 5))
plt.plot(t_infer, y_infer, label="true apogee")
plt.plot(t_infer, y_pred_physics, label="true apogee (physics)")
plt.plot(t_infer, y_pred, label="predicted apogee")
plt.xlabel("Time [s]")
plt.ylabel("Altitude [m]")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot error between predicted and true RK4
err_pred_physics = y_pred - y_pred_physics

plt.figure(figsize=(10, 5))
plt.plot(t_infer, err_pred_physics)
plt.xlabel("Time [s]")
plt.ylabel("Error [m]")
plt.show()

In [ ]:
# Plot predicted apogee error vs time
err_true = y_pred - y_infer

plt.figure(figsize=(10, 5))
plt.plot(t_infer, err_true)
plt.xlabel("Time [s]")
plt.ylabel("Predicted apogee error [m]")
plt.show()

In [ ]:
# Plot true apogee error vs time
err_physics = y_pred_physics - y_infer

plt.figure(figsize=(10, 5))
plt.plot(t_infer, err_physics)
plt.xlabel("Time [s]")
plt.ylabel("True RK4 apogee error [m]")
plt.show()

### C Generation

In [ ]:
# Generate C code

def export_weights_to_c(pt_file_path, header_file_path):
    # Load the PyTorch model state dictionary
    # Use map_location='cpu' in case it was saved on a GPU
    state_dict = torch.load(pt_file_path, map_location='cpu')
    
    with open(header_file_path, "w") as f:
        f.write("#ifndef _MODEL_WEIGHTS_H\n")
        f.write("#define _MODEL_WEIGHTS_H\n\n")
        f.write("// Auto-generated weights from PyTorch model\n\n")
        
        f.write(f"#define MAX_ALTITUDE {max_altitude:.1f}f\n")
        f.write(f"#define MAX_VELOCITY {max_velocity:.1f}f\n\n")
        
        for name, tensor in state_dict.items():
            # Clean up the variable name for C/C++ (replace dots with underscores)            
            c_name = name.replace('.', '_')
            
            # Convert to numpy and flatten to a 1D array
            flat_data = tensor.detach().numpy().flatten()
            
            # Write C array declaration
            f.write(f"const float {c_name}[{len(flat_data)}] = {{\n    ")

            # Format numbers safely as C floats
            values_str = ", ".join([f"{val:.7f}f" for val in flat_data])

            # Save
            f.write(values_str)    
            f.write("\n};\n")
            
            if len(tensor.shape) == 1:
                def_name = f"SIZE_{c_name.upper()}"
                f.write(f"#define {def_name} {len(flat_data)}\n")            

            f.write("\n")

        f.write("#endif // _MODEL_WEIGHTS_H\n")

    print("Saved!")

export_weights_to_c('airbrake_pinn.pt', './generated/model_weights.h')